<a href="https://colab.research.google.com/github/gkkg19/Deep-Learning/blob/main/MLP_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import time

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names

X_train_full, X_test_full, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_full = scaler.fit_transform(X_train_full)
X_test_full = scaler.transform(X_test_full)

print(f"Dataset: Train={X_train_full.shape[0]}, Test={X_test_full.shape[0]}, Features=30")

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def train_model(X_train, X_test, y_train, y_test):
    input_size = X_train.shape[1]
    hidden_size = 12

    np.random.seed(42)
    W1 = np.random.randn(input_size, hidden_size) * 0.1
    b1 = np.zeros((1, hidden_size))
    W2 = np.random.randn(hidden_size, 1) * 0.1
    b2 = np.zeros((1, 1))

    m, epochs, lr = X_train.shape[0], 2000, 0.5
    start_time = time.time()

    for epoch in range(epochs):
        H = sigmoid(X_train @ W1 + b1)
        O = sigmoid(H @ W2 + b2)

        dO = (O - y_train.reshape(-1, 1)) / m * O * (1 - O)
        dH = (dO @ W2.T) * H * (1 - H)

        dW2 = H.T @ dO
        dW1 = X_train.T @ dH

        # NO REGULARIZATION - pure gradients
        W2 -= lr * dW2
        W1 -= lr * dW1
        b2 -= lr * np.sum(dO, axis=0, keepdims=True)
        b1 -= lr * np.sum(dH, axis=0, keepdims=True)

    train_time = time.time() - start_time

    H_train = sigmoid(X_train @ W1 + b1)
    O_train = sigmoid(H_train @ W2 + b2)
    train_acc = accuracy_score(y_train, np.round(O_train))

    H_test = sigmoid(X_test @ W1 + b1)
    O_test = sigmoid(H_test @ W2 + b2)
    test_acc = accuracy_score(y_test, np.round(O_test))

    return {
        'train_acc': train_acc, 'test_acc': test_acc, 'time': train_time,
        'W1': W1, 'feature_importance': np.abs(W1).mean(axis=1)
    }

# Train NO REGULARIZATION model
print("\n=== NO REGULARIZATION  ===")
results = train_model(X_train_full, X_test_full, y_train, y_test)

# Display ALL feature weights
print("\n=== FEATURE WEIGHTS  ===")
importance = results['feature_importance']

print("\n" + "="*100)
print(f"{'Idx':<4} {'Feature Name':<45} {'Abs Weight (Importance)':<15}")
print("-"*100)

for idx in range(30):
    print(f"{idx:<4} {feature_names[idx]:<45} {importance[idx]:<15.4f}")

print("\n" + "="*100)
print(f"Train Accuracy: {results['train_acc']:.4f}")
print(f"Test Accuracy:  {results['test_acc']:.4f}")
print(f"Training Time:  {results['time']:.2f}s")



Dataset: Train=455, Test=114, Features=30

=== NO REGULARIZATION  ===

=== FEATURE WEIGHTS  ===

Idx  Feature Name                                  Abs Weight (Importance)
----------------------------------------------------------------------------------------------------
0    mean radius                                   0.1606         
1    mean texture                                  0.2584         
2    mean perimeter                                0.1646         
3    mean area                                     0.1981         
4    mean smoothness                               0.0808         
5    mean compactness                              0.0893         
6    mean concavity                                0.1395         
7    mean concave points                           0.1700         
8    mean symmetry                                 0.0741         
9    mean fractal dimension                        0.1421         
10   radius error                                  0.2362